# modular-diffusion: small engines that each learned one thing alone, served by place and thing addresses on the image tokens

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.12.0/colab/toy_collective.ipynb)

Synthetic scenes (`geolip.modular_diffusion.scenes`): a 48 x 48 canvas cut into a 3 x 3 grid, each cell holding one of nine things in one of nine colors or nothing, on a flat gray ground (bed F) or on one tint a scene with a smooth texture (bed C). Every image token carries its place digits and every request (one thing in one cell) six digits; the gate is declared, and every token has exactly one owner (`frame`). Small engines (width 128, 2 heads, 3 blocks) are trained ALONE, each on scenes holding one thing (`collective`): addressed thing parts that see only their own cell and the cells no other request owns, an addressed base, whole-picture parts, the empty-scene and the unconditional engines, and whole models trained on composed or on one-thing scenes, in two read forms (the address codes added to the token embeddings, or the request as a token of the sequence placed by an anchored rotary). The rows: the collective served by the address gate (R1, R1r), the whole-picture parts by the gate (R2) and by the published sums over an empty (R3) or an unconditional background (R4), the whole models (R5a, R5ar, R5b, R6), and real scenes (REAL), each sampled on 500 request lists of 1, 2, 3, 4, 6 and 9 things and judged cell by cell (`scene_judge`, `scene_reads`); the controls, the exactness reads (an engine exchanged for its twin, an engine added), the leak, the wrong-address read, the dose (1,500 and 6,000 steps) and an address never trained (the center cell left out of training).

1. install at a release tag and print the environment;
2. the judges of both beds trained first (their ceilings printed); a smoke of every job kind on the card; the seconds a step with 1, 2 and 4 trainings at once and the number of runners with the best throughput; R5b's shape and steps; the sampling time; the job list and its projected minutes; the smoke shipped;
3. the job list through several runners over one roster (jobs that read other jobs wait for them); each finished job folder shipped at once (weights included); a progress line a minute; the summary; the machine is given back only when every finished job and the report were shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`. On a new machine, finished jobs are fetched back from the Hub with their files, so nothing runs twice.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[hub] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.12.0"
import os, sys, time, json, math, gc, shutil, subprocess, threading
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 12, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import huggingface_hub
print("torch", torch.__version__, "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/toy_collective"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

First the two judges (one a bed, 2,000 steps on crops of fresh real scenes): their ceilings on 4,500 held-out crops are printed, and the cell stops when a ceiling is under 0.99 (control a), before any long run. Each judge folder is a finished job of the list (C3 reuses it) and is shipped. Then a smoke of every job kind at tiny settings on this card, the seconds a step of the engine's shape with 1, 2 and 4 short trainings running at the same time (as processes; the number of runners is the count with the best throughput), the seconds a step of R5b's shape (its steps: the ten addressed parts' training seconds over its seconds a step, a multiple of 500, at least 3,000), and the sampling time of the collective at n = 9; then the job list with its projection; the smoke folder is shipped.

In [ ]:
# C2 - the judges, the smoke, the timings, the job list, the projection
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import hub, run, jobs, scenes
from geolip.modular_diffusion import collective as C
from geolip.modular_diffusion.experiments import toy_collective as T
RUNS, SMOKE = f"{OUT}/runs", f"{OUT}/smoke"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/conditioning/toy_collective"
NOT_SHIPPED = ("SHIPPED", "FROM_HUB", "checkpoint.safetensors", "checkpoint.json")
DEVICE = "cuda"
JUDGE_STEPS, JUDGE_EXTRA = 2000, ""
PART_STEPS, DOSE_STEPS, MODEL_STEPS = 3000, (1500, 6000), 30000
R5B_FLOOR, R5B_ROUND = 3000, 500
TIMING_STEPS, TIMING_FROM, TIMING_COUNTS = 120, 20, (1, 2, 4)
SAMPLE_LISTS, SAMPLE_STEPS = 500, 50
SMOKE_ARGS = ("steps=2 part_steps=2 model_steps=2 batch=4 heldout=8 every=1 count=8 ns=1,2,9 sample_steps=2 chunk=0 "
              "exact_lists=8 leak_scenes=4 judge_steps=2 judge_batch=8 save_every=0")
os.makedirs(RUNS, exist_ok=True)
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only

def fetch_back(tags, needs):
    # finished jobs come back from the Hub with their files, so nothing runs twice; a job comes back only when every
    # job it reads came back (or finished) too
    if not REPO:
        return
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        if not all(mark(n, "DONE") for n in needs.get(t, [])):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

def ship_job(tag, folder):
    if REPO and hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"toy collective: {tag}", ignore=NOT_SHIPPED):
        open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")
        return True
    return False

# the judges first: control (a)'s first half before any long run
fetch_back(["judge_F", "judge_C"], {})
CEILING = {}
for bed in ("F", "C"):
    d = f"{RUNS}/judge_{bed}"
    if not mark(f"judge_{bed}", "DONE"):
        shutil.rmtree(d, ignore_errors=True)
        run.main("toy_collective", f"variant=judge bed={bed} seed=0 judge_steps={JUDGE_STEPS} device={DEVICE}{JUDGE_EXTRA}".split() + [f"out={d}"])
        open(f"{d}/DONE", "w").write("trained in C2\n")
    j = json.load(open(f"{d}/result.json", encoding="utf-8"))["judge"]
    CEILING[bed] = j["ceiling"]
    print(f"judge {bed}: things {j['ceiling']['thing_rate']['mean']:.4f}, colors {j['ceiling']['color_rate']['mean']:.4f} "
          f"on {j['ceiling']['crops']} held-out crops; {j['train_seconds']:.0f} s")
    if REPO and not (mark(f"judge_{bed}", "SHIPPED") or mark(f"judge_{bed}", "FROM_HUB")):
        assert ship_job(f"judge_{bed}", d), f"the judge {bed} was not shipped: check the token and the repository"
assert all(CEILING[b]["thing_rate"]["mean"] >= 0.99 and CEILING[b]["color_rate"]["mean"] >= 0.99 for b in CEILING) or JUDGE_STEPS < 2000, \
    "control (a): a judge's ceiling is under 0.99; stop here"

# the smoke: every job kind at tiny settings on this card
shutil.rmtree(SMOKE, ignore_errors=True)
SM = SMOKE_ARGS.split() + [f"device={DEVICE}"]
def smoke(name, extra):
    t0 = time.time()
    r = run.main("toy_collective", SM + extra.split() + [f"out={SMOKE}/{name}"])
    print(f"smoke {name}: {time.time() - t0:.0f} s", flush=True)
    return r
smoke("judge_F", "variant=judge bed=F")
for kind, classes, seed, read, cells in (("addressed_thing", "0-8", 0, "embedding", ""), ("addressed_base", "", 0, "embedding", ""),
                                         ("addressed_thing", "0-8", 0, "rotary", ""), ("addressed_base", "", 0, "rotary", ""),
                                         ("whole_thing", "0-8", 0, "embedding", ""), ("whole_empty", "", 0, "embedding", ""),
                                         ("whole_unconditional", "", 0, "embedding", ""), ("model_mix", "", 0, "embedding", ""),
                                         ("model_mix", "", 0, "rotary", ""), ("model_one", "", 0, "embedding", ""),
                                         ("addressed_thing", "0-8", 1, "embedding", ""), ("addressed_thing", "0-8", 1, "rotary", ""),
                                         ("whole_thing", "0-8", 1, "embedding", ""), ("addressed_thing", "0-8", 0, "embedding", "01235678")):
    smoke(f"parts_{kind}_{seed}_{read}{'_' + cells if cells else ''}", f"variant=parts kind={kind} classes={classes} seed={seed} read={read} bed=F"
          + (f" cells={cells}" if cells else ""))
SP = f"parts={SMOKE} judge={SMOKE}/judge_F bed=F"
for row in ("REAL", "R1", "R1r", "R2", "R3", "R4", "R5a", "R5ar", "R6"):
    r = smoke(f"row_{row}", f"variant=compose row={row} {SP}")
smoke("row_R1_center", f"variant=compose row=R1 lists=center part_cells=01235678 {SP}")
c = smoke("controls_R1", f"variant=controls row=R1 {SP}")["controls"]
assert c["d"]["holds"] and c["e"]["holds"] and c["g"]["holds"], "controls (d), (e) or (g) failed in the smoke"
assert smoke("controls_R1r", f"variant=controls row=R1r {SP}")["controls"]["d"]["holds"], "control (d) on the rotary collective"
ex = smoke("exact", f"variant=exact rows=R1,R1r,R2,R3 {SP}")["exact"]
assert ex["R1"]["exchange_exact"] and ex["R1"]["snap_in_exact"] and ex["R1r"]["exchange_exact"], "the exactness reads in the smoke"
print("the smoke's exchange on R2 and R3 changes pixels outside:", ex["R2"]["largest_outside"], ex["R3"]["largest_outside"])
smoke("leak", f"variant=leak {SP}")
smoke("wrong", f"variant=wrong {SP}")

# the timings: seconds a step of the engine's shape with 1, 2 and 4 trainings at once (processes on this card)
def timing(k, shape=""):
    procs, outs = [], []
    for i in range(k):
        o = f"{SMOKE}/timing_{k}x_{i}" + ("_" + shape.replace(" ", "_").replace("=", "") if shape else "")
        shutil.rmtree(o, ignore_errors=True)
        cmd = [sys.executable, "-m", "geolip.modular_diffusion.run", "toy_collective", "variant=timing", "kind=addressed_thing",
               f"steps={TIMING_STEPS}", f"from_step={TIMING_FROM}", "heldout=256", f"device={DEVICE}", f"out={o}"] + shape.split()
        procs.append(subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
        outs.append(o)
    for p in procs:
        p.wait()
    secs = [json.load(open(f"{o}/result.json", encoding="utf-8"))["timing"]["seconds_a_step"] for o in outs]
    return sum(secs) / len(secs)
SEC = {k: timing(k) for k in TIMING_COUNTS}
THROUGHPUT = {k: k / SEC[k] for k in TIMING_COUNTS}            # trainings x steps a second
for k in TIMING_COUNTS:
    print(f"{k} training(s) at once: {SEC[k]:.4f} s a step each; throughput {THROUGHPUT[k]:.2f} steps a second")
RUNNERS_MEASURED = max(TIMING_COUNTS, key=lambda k: THROUGHPUT[k])
TEN = 10 * C.shape_numbers(128, 2, 3)
R5B_DEPTH = C.depth_to_reach(TEN, 256, 4)
SEC_R5B = timing(1, f"dim=256 heads=4 depth={R5B_DEPTH}")
R5B_STEPS = max(R5B_FLOOR, int(math.ceil(10 * PART_STEPS * SEC[1] / SEC_R5B / R5B_ROUND) * R5B_ROUND))
print(f"R5b: width 256, 4 heads, depth {R5B_DEPTH} ({C.shape_numbers(256, 4, R5B_DEPTH):,} numbers against the ten parts' {TEN:,}); "
      f"{SEC_R5B:.4f} s a step alone; {R5B_STEPS:,} steps (the ten parts' {10 * PART_STEPS * SEC[1]:.0f} s)")
things = {k: C.born("addressed_thing", k, 0).to(DEVICE).eval() for k in range(9)}
col = C.Collective(things, C.born("addressed_base", None, 0).to(DEVICE).eval(), addressed=True)
rl = scenes.request_lists(9, SAMPLE_LISTS)
t0 = time.time()
C.sample(lambda z, t, c, k: col.answer(z, t, c, k, "gate"), rl.cells, rl.colors, 7009, SAMPLE_STEPS, DEVICE, 128)
if torch.cuda.is_available():
    torch.cuda.synchronize()
SAMPLE_S9 = time.time() - t0
print(f"sampling R1 at n = 9: {SAMPLE_LISTS} lists, {SAMPLE_STEPS} steps in {SAMPLE_S9:.0f} s")
del things, col
gc.collect()

# the job list: (tag, arguments, needs, seconds at one runner)
def job_list(ps, dose, ms, r5b_depth, r5b_steps):
    J = []
    def add(tag, args, needs=(), cost=60.0):
        J.append((tag, args, list(needs), float(cost)))
    for bed in ("F", "C"):
        add(f"judge_{bed}", f"variant=judge bed={bed} seed=0 judge_steps={JUDGE_STEPS} device={DEVICE}{JUDGE_EXTRA}", cost=600)
    def parts(name, bed, seed, kind, classes, steps, read="embedding", shape="", cells="", sec=None):
        tagp = f"{name}_{bed}_s{seed}"
        n = 9 if classes else 1
        add(tagp, f"variant=parts kind={kind} classes={classes} bed={bed} seed={seed} steps={steps} read={read} device={DEVICE}"
            + (f" {shape}" if shape else "") + (f" cells={cells}" if cells else ""), cost=n * steps * (sec or SEC[1]) + 60)
        return tagp
    B5 = f"dim=256 heads=4 depth={r5b_depth}"
    for bed in ("F", "C"):
        for seed in (0, 1):
            parts("at", bed, seed, "addressed_thing", "0-8", ps); parts("ab", bed, seed, "addressed_base", "", ps)
            parts("atr", bed, seed, "addressed_thing", "0-8", ps, "rotary"); parts("abr", bed, seed, "addressed_base", "", ps, "rotary")
            parts("wt", bed, seed, "whole_thing", "0-8", ps); parts("we", bed, seed, "whole_empty", "", ps)
            if bed == "F":
                parts("wu", bed, seed, "whole_unconditional", "", ps)
            parts("mm", bed, seed, "model_mix", "", ms); parts("mmr", bed, seed, "model_mix", "", ms, "rotary")
            parts("mmb", bed, seed, "model_mix", "", r5b_steps, shape=B5, sec=SEC_R5B)
            if bed == "F":
                parts("mo", bed, seed, "model_one", "", ms)
                parts("atx", bed, seed, "addressed_thing", "0-8", ps, cells="01235678")
                parts("atxr", bed, seed, "addressed_thing", "0-8", ps, "rotary", cells="01235678")
    for steps in dose:
        for read, nm in (("embedding", ""), ("rotary", "r")):
            add(f"dose{steps}_at{nm}_F_s0", f"variant=parts kind=addressed_thing classes=0-8 bed=F seed=0 steps={steps} read={read} device={DEVICE}",
                cost=9 * steps * SEC[1] + 60)
            add(f"dose{steps}_ab{nm}_F_s0", f"variant=parts kind=addressed_base classes= bed=F seed=0 steps={steps} read={read} device={DEVICE}",
                cost=steps * SEC[1] + 60)
    row_cost = 3.0 * SAMPLE_S9 * 25 / 9 + 120
    NEED = {"R1": ("at", "ab"), "R1r": ("atr", "abr"), "R2": ("wt", "we"), "R3": ("wt", "we"), "R4": ("wt", "wu"),
            "R5a": ("mm",), "R5ar": ("mmr",), "R5b": ("mmb",), "R6": ("mo",)}
    for bed in ("F", "C"):
        common = f"bed={bed} parts={RUNS} judge={RUNS}/judge_{bed} part_steps={ps} model_steps={ms} device={DEVICE}"
        add(f"REAL_{bed}", f"variant=compose row=REAL seed=0 {common}", [f"judge_{bed}"], 300)
        for seed in (0, 1):
            for row in ("R1", "R1r", "R2", "R3", "R5a", "R5ar", "R5b") + (("R4", "R6") if bed == "F" else ()):
                shape = f" model_steps={r5b_steps} model_dim=256 model_heads=4 model_depth={r5b_depth}" if row == "R5b" else ""
                add(f"{row}_{bed}_s{seed}", f"variant=compose row={row} seed={seed} {common}{shape}",
                    [f"judge_{bed}"] + [f"{n}_{bed}_s{seed}" for n in NEED[row]], row_cost)
            add(f"leak_{bed}_s{seed}", f"variant=leak seed={seed} {common}", [f"wt_{bed}_s{seed}", f"we_{bed}_s{seed}"], 600)
        add(f"controls_{bed}", f"variant=controls row=R1 seed=0 {common}",
            [f"judge_{bed}"] + [f"{n}_{bed}_s0" for n in ("at", "ab", "wt", "we")], 3 * row_cost)
        add(f"controls_r_{bed}", f"variant=controls row=R1r seed=0 {common}",
            [f"judge_{bed}"] + [f"{n}_{bed}_s0" for n in ("atr", "abr")], 3 * row_cost)
        add(f"exact_{bed}", f"variant=exact rows=R1,R1r,R2,R3 seed=0 twin_seed=1 {common}",
            [f"{n}_{bed}_s{s}" for n in ("at", "ab", "atr", "abr", "wt", "we") for s in (0, 1)], 1200)
    commonF = f"bed=F parts={RUNS} judge={RUNS}/judge_F model_steps={ms} device={DEVICE}"
    for seed in (0, 1):
        add(f"wrong_F_s{seed}", f"variant=wrong seed={seed} part_steps={ps} {commonF}",
            ["judge_F", f"at_F_s{seed}", f"ab_F_s{seed}"], 4 * row_cost)
        for row, nm in (("R1", ""), ("R1r", "r")):
            for lists in ("center", "others"):
                add(f"never{nm}_{lists}_F_s{seed}", f"variant=compose row={row} seed={seed} lists={lists} part_cells=01235678 part_steps={ps} {commonF}",
                    ["judge_F", f"atx{nm}_F_s{seed}", f"ab{nm}_F_s{seed}"], 300)
    for steps in dose:
        for row, nm in (("R1", ""), ("R1r", "r")):
            add(f"dose{steps}_{row}_F_s0", f"variant=compose row={row} seed=0 part_steps={steps} {commonF}",
                ["judge_F", f"dose{steps}_at{nm}_F_s0", f"dose{steps}_ab{nm}_F_s0"], row_cost)
    return J

JOBS = job_list(PART_STEPS, DOSE_STEPS, MODEL_STEPS, R5B_DEPTH, R5B_STEPS)
COST = {t: c for t, _, _, c in JOBS}
pos = {t: i for i, (t, *_) in enumerate(JOBS)}
assert len(pos) == len(JOBS) and all(n in pos and pos[n] < pos[t] for t, _, ns, _ in JOBS for n in ns), "the job list's needs"
speedup = RUNNERS_MEASURED * SEC[1] / SEC[RUNNERS_MEASURED]
INSTRUMENTS = {"ceiling": CEILING, "seconds_a_step": SEC, "throughput": THROUGHPUT, "runners_measured": RUNNERS_MEASURED,
               "r5b": {"depth": R5B_DEPTH, "steps": R5B_STEPS, "seconds_a_step": SEC_R5B, "numbers": C.shape_numbers(256, 4, R5B_DEPTH),
                       "ten_parts_numbers": TEN}, "sample_seconds_n9": SAMPLE_S9, "projected_seconds_one_runner": COST,
               "card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "package": MD.__version__,
               "torch": torch.__version__}
json.dump(INSTRUMENTS, open(f"{SMOKE}/instruments.json", "w", encoding="utf-8"), indent=1)
print(f"the job list: {len(JOBS)} jobs, about {sum(COST.values()) / speedup / 60:.0f} projected minutes on this card with "
      f"{RUNNERS_MEASURED} runner(s) (throughput {THROUGHPUT[RUNNERS_MEASURED]:.2f} steps a second)")
if REPO:
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke", "toy collective: smoke run", ignore=NOT_SHIPPED), (
        "the smoke folder was not shipped: check the token and the repository before the long run")

RUNNERS runners (by default the measured best count) work one roster: the judges, the parts (two seeds, both read forms, the whole models, the dose, the address never trained), then the rows and the reads, each of which waits for the jobs it reads (`needs=`). Every job runs with deterministic algorithms on, and nothing about a job's result depends on how many runners there are. Each finished job folder is shipped at once to `REPO` with its weights (checkpoints are left out), with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). At the end the summary, the roster, its logs and the instruments are shipped to `_report`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the job list through several runners, each finished job shipped, a progress line a minute, the summary, the machine
RUNNERS = RUNNERS_MEASURED         # the number of runners (the measured best; set a number to override)
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
EXTRA = ""                         # more arguments for every job (empty in a real run)
PROGRESS_S = 60
THREADS = max(2, min(4, (os.cpu_count() or 4) // RUNNERS))
tags = [t for t, *_ in JOBS]
needs = {t: n for t, _, n, _ in JOBS}
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | toy_collective {a} det=on{EXTRA}"
                       + (f" | needs={','.join(n)}" if n else "") for t, a, n, _ in JOBS) + "\nEND\n")
fetch_back(tags, needs)
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and REPO and not ship_job(tag, folder):
        print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
speed = RUNNERS * SEC[1] / SEC.get(RUNNERS, SEC[1])
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / speed / 60:.0f} min to run "
      f"with {RUNNERS} runner(s)")
runners = [threading.Thread(target=jobs.run_roster, daemon=True,
                            kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster_{k}.log", card_room_mib=None,
                                        ram_free_gb=4.0, max_hours=40.0, claim=True, after_job=after_job, wait_s=5.0,
                                        env={"OMP_NUM_THREADS": str(THREADS), "MKL_NUM_THREADS": str(THREADS), "HF_HUB_OFFLINE": "1"}))
           for k in range(RUNNERS)]
for th in runners:
    th.start()
while any(th.is_alive() for th in runners):
    time.sleep(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else speed
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

summary = T.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ["roster.txt"] + [f"roster_{k}.log" for k in range(RUNNERS)]:
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{f.replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report", "toy collective: summary, roster and instruments") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none")
if RELEASE and shipped and not unshipped:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else "RELEASE is False")